# Phase 2 pipeline execution

## Step 1: data cleaning (`src/clean_items.py`)
Only multiple-choice and true/false questions are kept. The raw extraction has known errors (see `PROJECT_LOG.md`); every fix is a logged rule and the raw file is never changed. The question and the answer end up in two tables joined by `item_id`.


In [ ]:
import sys, pathlib
import pandas as pd
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import clean_items

raw = pd.read_csv(ROOT / 'data/raw/items.csv')
clean = clean_items.finalise(clean_items.run(raw))
log = pd.DataFrame(clean_items.LOG)
questions, answers = clean_items.split_qa(clean)
print(f'raw {raw.shape}  ->  MCQ/TF rows {clean.shape}  ->  usable questions {questions.shape}, answers {answers.shape}')

### What each rule changed

In [ ]:
log.groupby('rule').size().rename('rows').to_frame()


### Which rows can be used for modelling, and why the others cannot


In [ ]:
clean.exclude_reason.replace('', 'USABLE').value_counts().to_frame('rows')


In [ ]:
pd.crosstab(questions.class_level, questions.item_type, margins=True)


### Save
`python src/clean_items.py` does the same from the command line.


In [ ]:
out = ROOT / 'data/processed'
out.mkdir(exist_ok=True)
clean.to_csv(out / 'items_clean.csv', index=False)
questions.to_csv(out / 'questions.csv', index=False)
answers.to_csv(out / 'answers.csv', index=False)
log.to_csv(out / 'cleaning_log.csv', index=False)


## Step 2: merge questions and answers on item_id

In [ ]:
from src.merge_qa import merge_qa
q=pd.read_csv('data/processed/questions.csv'); a=pd.read_csv('data/processed/answers.csv')
qa=merge_qa(q[q.item_id.isin(a.item_id)], a)
print(len(q),len(a),len(qa))

## Step 3: candidate table, grouped split, pipeline fitted on train only

In [ ]:
from src.candidates import build_candidates, assign_split
from src.pipeline_builder import build_preprocessor, FEATURES
c=pd.read_csv('data/processed/candidates.csv')
tr,te=c[c.split=='train'],c[c.split=='test']
pre=build_preprocessor().fit(tr[FEATURES])
Xt=pre.transform(tr[FEATURES]); print(Xt.shape, pre.transform(te[FEATURES]).shape)
c.split.value_counts()

## Step 4: the eight figures

In [ ]:
from src import figures
figures.main()
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/fig*.png')): display(Image(f, width=700))